# Parameter sensitivity — which uncertainty to verify first

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

Uncertainty propagation (notebook 06) gives prediction *bands*; sensitivity analysis attributes that variance to individual parameters. Because each IIV-bearing parameter is sampled independently (lognormal), the standardized regression coefficient (SRC) of a parameter equals its correlation with the target and the squared SRCs partition the explained variance.

The payoff is curation triage: the spec's highest-leverage contribution is verifying records against the source PDF (§9), and this says *which* parameter's uncertainty actually moves the survival prediction.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import onkos

ds = onkos.load()
res = onkos.sensitivity(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'NSCLC', 'line': 'first'},
                        target='median_os_weeks', n=500, seed=0)
print(f'target={res.target}  first-order R^2={res.r_squared:.2f}')
for p in res.indices:
    print(f'  {p.symbol:<8} CV {p.iiv_cv_percent:>3.0f}%  SRC {p.src:+.3f}  contribution {p.contribution*100:.0f}%')
assert abs(sum(p.contribution for p in res.indices) - 1.0) < 1e-6

In [ ]:
# Higher CV does not imply higher influence: influence = variability x effect strength.
# Here the kill rate kD (CV 89%) dominates median-OS variance over lambda (CV 96%).
dom = res.dominant
print('dominant parameter:', dom.symbol, '->', f'{dom.contribution*100:.0f}% of variance')
assert dom.symbol == 'kD'

rows = list(reversed(res.indices))
plt.barh([p.symbol for p in rows], [p.contribution*100 for p in rows],
         color=['#2f855a' if p.src > 0 else '#c53030' for p in rows])
plt.xlabel('contribution to median-OS variance (%)'); plt.title('sensitivity tornado');

In [ ]:
# The ranking is target-dependent and direction-aware.
for target in ['median_os_weeks', 'week8_relative_change', 'depth_of_response']:
    r = onkos.sensitivity(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'NSCLC', 'line': 'first'},
                          target=target, n=300, seed=1)
    top = r.dominant
    print(f'{target:<24} -> {top.symbol} (SRC {top.src:+.2f}, {top.contribution*100:.0f}%)')